# Manual técnico do sistema de treinamento médico com multiagentes em Amazon Bedrock e Django
Documento técnico consolidado da solução de treinamento de propagandistas para interação com médicos. O sistema utiliza Django como camada de aplicação, persistência e interface, e Amazon Bedrock Agents como camada de orquestração multiagente e recuperação factual por Knowledge Base.


## 1. Objetivo funcional
O objetivo principal é treinar propagandistas em assuntos médicos, especialidades, cenários de visita, medicamentos, objeções e formas de abordagem com médicos reais. O sistema precisa sustentar dois modos complementares:
1. **Treinamento orientado**: explicação didática, simulação de conversa, prática de objeções e coaching.
2. **Consulta factual**: recuperação de fatos específicos armazenados na Knowledge Base, como campos de documentos, dados de bula, responsáveis técnicos, evidências e informações de suporte.


## 2. Princípios de arquitetura
### 2.1 Camada Bedrock
A camada Bedrock executa a inteligência multiagente. O desenho adotado é composto por um agente supervisor e quatro colaboradores:
- **consulta**: recuperação factual, RAG e uso da Knowledge Base.
- **síntese**: resposta final em linguagem natural, didática ou simulada.
- **compliance**: validação promocional, científica e de segurança.
- **avaliação**: feedback incremental e avaliação consolidada do propagandista.

### 2.2 Camada Django
A camada Django sustenta a aplicação e não substitui a orquestração Bedrock. As responsabilidades principais são:
- autenticação e interface web;
- persistência das tabelas de catálogo e conversa;
- criação de sessões de simulação;
- envio de mensagens via WebSocket;
- armazenamento de turns, execuções de agentes e avaliações;
- composição de contexto e invocação do alias do supervisor Bedrock.


## 3. Conceito de versões, draft e aliases no Bedrock
Amazon Bedrock Agents trabalha com **working draft** e **aliases**. O rascunho operacional é o `DRAFT`. O alias de teste `TSTALIASID` aponta para o `DRAFT`. Aliases normais apontam para versões publicadas e imutáveis. Alterações de instrução, colaboração e knowledge base são feitas no draft; depois o agente precisa ser preparado e publicado por meio de nova versão/alias. Esse conceito é central para entender por que mudanças locais em `prompting.py` não alteram imediatamente o comportamento dos agentes já publicados. Conforme a documentação da AWS, o agente nasce com `DRAFT` e `TSTALIASID`, e aliases estáveis devem apontar para versões publicadas. [AWS Bedrock deploy/aliases](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-agent.html)


## 4. Componentes do sistema
### 4.1 Agentes Bedrock
**Supervisor**
- coordena os colaboradores;
- decide quando usar consulta, síntese, compliance e avaliação;
- devolve o contrato final para a aplicação.

**Consulta**
- recupera fatos e evidências;
- usa a Knowledge Base associada ao agente;
- deve trabalhar por intenção da pergunta, entidade principal, atributo procurado e reformulação iterativa.

**Síntese**
- transforma evidência e contexto em resposta didática ou simulação de médico;
- precisa preservar fatos vindos da consulta.

**Compliance**
- revisa claims, extrapolações e riscos regulatórios.

**Avaliação**
- mede a qualidade do desempenho do propagandista por turno e na consolidação final.

### 4.2 Aplicação Django
**Catálogo**
- blueprints de simulação;
- personas;
- cenários;
- especialidades;
- políticas;
- instruções;
- contratos de saída;
- rubricas de avaliação.

**Conversas**
- sessão de simulação;
- turnos;
- execuções de agentes;
- avaliação por turno.

**Realtime**
- WebSocket via ASGI/Channels/Daphne.


## 5. Responsabilidade por camada
| Camada | Responsabilidade |
|---|---|
| AWS Bedrock Agents | Orquestração multiagente, collaboration, Knowledge Base, execução do raciocínio agentic |
| IAM | Permissões de colaboração entre aliases, `bedrock:GetAgentAlias` e `bedrock:InvokeAgent` |
| Django | Interface, persistência, catálogo, sessão, WebSocket, invocação do supervisor |
| Banco Django | Estado da sessão, turns, execuções e avaliações |
| Front-end Django | Seleção de blueprint/persona/cenário/especialidade e interação do usuário |


## 6. Variáveis de ambiente principais
Formato utilizado em PowerShell:

```powershell
$env:AWS_ACCESS_KEY_ID="..."
$env:AWS_SECRET_ACCESS_KEY="..."
$env:AWS_REGION="us-east-1"
$env:BEDROCK_KB_ID="BJWEDBPNJH"
$env:BEDROCK_AGENT_ROLE_ARN="arn:aws:iam::<ACCOUNT_ID>:role/BedrockAgentsServiceRole"

$env:CONSULTA_AGENT_ID="MLP8GVTSYX"
$env:SINTESE_AGENT_ID="WNPST1YIUL"
$env:COMPLIANCE_AGENT_ID="T1S2H8JUIA"
$env:AVALIACAO_AGENT_ID="NSHP4AAEKT"
$env:SUPERVISOR_AGENT_ID="JO0SG5Z8EI"

$env:CONSULTA_ALIAS_ID="Q7IB3UZ90R"
$env:SINTESE_ALIAS_ID="O4WXGEL5BJ"
$env:COMPLIANCE_ALIAS_ID="EOLTUUKZY6"
$env:AVALIACAO_ALIAS_ID="SE4UGP8FTG"
$env:SUPERVISOR_ALIAS_ID="LB6TMVZIHT"
```

Em Django, variáveis adicionais relevantes:

```powershell
$env:DJANGO_SECRET_KEY="<segredo>"
$env:DJANGO_DEBUG="True"
$env:DJANGO_ALLOWED_HOSTS="127.0.0.1,localhost"
```


## 7. Estrutura lógica do projeto Django
### 7.1 `core` / `config`
- configuração global do projeto;
- leitura de ambiente;
- manifesto Bedrock;
- `asgi.py`;
- `urls.py`.

### 7.2 `apps/catalog`
- modelos de catálogo da simulação;
- tabelas que persistem blueprint, persona, cenário, especialidade, políticas e instruções;
- telas iniciais e seleção da simulação.

### 7.3 `apps/conversations`
- `ConversationSession`;
- `ConversationTurn`;
- `AgentRun`;
- persistência de estado e avaliações.

### 7.4 `apps/agents`
- integração Bedrock;
- `prompting.py` para composição das instruções dos agentes;
- `services.py` para invocação do supervisor;
- provisionamento e binding do time Bedrock.

### 7.5 `apps/realtime`
- consumer WebSocket;
- recepção de mensagens do usuário;
- envio do resultado do supervisor à interface.

### 7.6 `apps/api`
- endpoints HTTP auxiliares.


## 8. Modelo de dados recomendado
### 8.1 Catálogo
- `SimulationBlueprint`
- `Persona`
- `Scenario`
- `Specialty`
- `Policy`
- `Instruction`
- `OutputContract`
- `EvaluationRubric`

### 8.2 Conversa
- `ConversationSession`
- `ConversationTurn`
- `AgentRun`

### 8.3 Observação operacional
Personas, cenários, especialidades e instruções devem permanecer em tabelas, não hardcoded em JSON estático. O JSON de prompt deve ser um **artefato derivado** das tabelas. Esse desenho facilita CRUD futuro e governança.


## 9. Fluxo de criação e publicação dos agentes
### 9.1 Criação dos agentes
Para cada papel, foi criado um agente Bedrock nativo com `create-agent`:
- consulta
- síntese
- compliance
- avaliação
- supervisor

### 9.2 Associação da Knowledge Base
A Knowledge Base foi associada ao agente de consulta. Esse agente concentra o papel de RAG.

### 9.3 Aliases
Foram utilizados alias de teste (`TSTALIASID`) e aliases publicados. Na prática, a colaboração entre supervisor e agentes colaboradores exigiu aliases publicados normais, não o alias de teste de outro agente.

### 9.4 Colaboração
O supervisor recebeu colaboradores por meio de `associate-agent-collaborator` / `update-agent-collaborator`.

### 9.5 Permissões
A role `BedrockAgentsServiceRole` recebeu uma policy inline adicional para permitir `bedrock:GetAgentAlias` e `bedrock:InvokeAgent` sobre os aliases colaboradores. Sem isso, a colaboração falha.


## 10. Sequência operacional que se mostrou correta
1. criar ou atualizar o agente no draft;
2. associar ou atualizar a Knowledge Base no agente de consulta;
3. `prepare-agent`;
4. criar alias novo para gerar versão publicada;
5. atualizar a policy IAM com o ARN do alias novo;
6. atualizar o colaborador do supervisor para o alias novo;
7. `prepare-agent` do supervisor;
8. atualizar o manifesto Django;
9. criar sessão nova no Django.


## 11. Manifesto Bedrock na aplicação
O manifesto liga o blueprint da simulação aos IDs e aliases dos agentes. Exemplo conceitual:

```json
{
  "blueprints": {
    "default": {
      "supervisor": {"agent_id": "JO0SG5Z8EI", "alias_id": "LB6TMVZIHT"},
      "collaborators": {
        "consultation": {"agent_id": "MLP8GVTSYX", "alias_id": "Q7IB3UZ90R"},
        "synthesis": {"agent_id": "WNPST1YIUL", "alias_id": "O4WXGEL5BJ"},
        "compliance": {"agent_id": "T1S2H8JUIA", "alias_id": "EOLTUUKZY6"},
        "evaluation": {"agent_id": "NSHP4AAEKT", "alias_id": "SE4UGP8FTG"}
      },
      "knowledge_base_id": "BJWEDBPNJH"
    }
  }
}
```

Uma observação crítica: a sessão Django salva `bedrock_team` em `session_state`. Após qualquer mudança de alias ou manifesto, é necessário abrir uma **sessão nova**.


## 12. Fluxo de execução da aplicação
### 12.1 Fluxo funcional
1. usuário escolhe blueprint, persona, cenário e especialidade;
2. Django cria `ConversationSession`;
3. Django salva o binding Bedrock no `session_state`;
4. usuário envia mensagem por WebSocket;
5. o consumer chama `BedrockNativeSupervisorOrchestrator`;
6. o orquestrador monta contexto, histórico, bundle de catálogo e hints de retrieval;
7. o supervisor Bedrock é invocado pelo alias configurado;
8. o supervisor coordena consulta, síntese, compliance e avaliação;
9. a aplicação persiste o retorno em `ConversationTurn` e `AgentRun`;
10. a interface recebe resposta, avaliação e demais artefatos.

### 12.2 Fluxo lógico entre agentes
```text
Usuário
  -> Django WebSocket consumer
    -> Orchestrator local
      -> Supervisor Bedrock
         -> Consulta (quando há necessidade factual / documental)
         -> Síntese (didática / simulação)
         -> Compliance (validação)
         -> Avaliação (feedback)
      -> Django persiste retorno
  -> UI renderiza resposta e avaliação
```


## 13. Como os prompts são montados
### 13.1 Local de tuning
O tuning principal dos agentes fica em `apps/agents/prompting.py`.

### 13.2 Métodos de composição
- `compose_consultation_instruction(...)`
- `compose_synthesis_instruction(...)`
- `compose_compliance_instruction(...)`
- `compose_evaluation_instruction(...)`
- `compose_supervisor_instruction(...)`

### 13.3 Papel de `services.py`
`apps/agents/services.py` não substitui a instrução do agente publicado. Ele monta o contexto do turno, classifica a pergunta, deriva hints de recuperação e invoca o alias do supervisor. O tuning local do `services.py` melhora a mensagem enviada ao supervisor, mas a mudança de comportamento do agente Bedrock só se torna real após reprovisionamento do agente publicado.


## 14. Estratégia de tuning por agente
### 14.1 Consulta
Deve concentrar:
- recuperação factual;
- extração de entidade;
- extração de atributo;
- sinônimos;
- múltiplas reformulações de query;
- preservação literal do fato encontrado.

### 14.2 Síntese
Deve concentrar:
- explicação didática;
- simulação de fala do médico;
- encerramento com próximos passos.

### 14.3 Compliance
Deve concentrar:
- claims;
- aderência regulatória;
- reescrita segura.

### 14.4 Avaliação
Deve concentrar:
- clareza;
- condução da conversa;
- uso de evidência;
- tratamento de objeções;
- coaching.

### 14.5 Supervisor
Deve concentrar:
- política de orquestração;
- decisão entre consulta factual e resposta de treinamento;
- preservação de fatos da base quando encontrados;
- continuidade da conversa.


## 15. Problema atual em aberto
O principal problema técnico remanescente é este:
- o sistema de treinamento já funciona, com WebSocket, sessão, orquestração Bedrock e persistência;
- porém a consulta factual específica em Knowledge Base ainda não retorna com consistência alguns campos específicos, mesmo quando o conteúdo existe na base.

Hipótese operacional mais forte:
- o agente de consulta consegue usar a base, mas o retrieval ainda está falhando em recuperar atributos específicos de determinados documentos;
- o próximo passo é reforçar consulta iterativa por entidade + atributo + sinônimos e validar a ingestão/chunking da Knowledge Base.


## 16. Procedimento de tuning seguro no futuro
Quando houver ajuste de comportamento de agente, a sequência correta é:
1. editar `prompting.py`;
2. editar `services.py` se necessário;
3. atualizar o draft do agente Bedrock;
4. `prepare-agent`;
5. criar alias novo ou versão nova;
6. atualizar permissões IAM para o alias novo;
7. atualizar o colaborador do supervisor;
8. `prepare-agent` do supervisor;
9. atualizar o manifesto da aplicação;
10. criar sessão nova no Django.


## 17. Configuração Django resumida
### 17.1 Dependências principais
- Django
- Channels
- Daphne
- boto3

### 17.2 Execução local
Para WebSocket, a execução correta é ASGI com Daphne. Em desenvolvimento, o uso de `runserver` foi insuficiente para o fluxo completo de WebSocket; o caminho estável foi `daphne -p 8000 config.asgi:application`.

### 17.3 Migrações
A correção dos imports antigos e dos modelos eliminou dependências obsoletas como `PromptComponent` e `ConversationArtifact`. A aplicação passou a depender do catálogo persistido em tabelas.


## 18. Administração e seed de dados
Foi adotado seed de catálogo para popular personas, cenários, especialidades, políticas, instruções e blueprints. As telas iniciais passam a ler as opções do banco, não mais de templates estáticos soltos.


## 19. Recomendações finais de engenharia
- manter o Bedrock como camada agentic nativa;
- manter Django como camada de interface, persistência e controle de sessão;
- reduzir qualquer fallback que mascare erro real do supervisor;
- sempre testar colaborador isolado antes do supervisor quando houver suspeita de problema na Knowledge Base;
- sempre abrir nova sessão após alteração de alias/manifesto/binding.


## 20. Referências operacionais essenciais
- Deploy e aliases de agentes Bedrock: AWS mantém o conceito de `DRAFT`, `TSTALIASID` e aliases publicados. [Deploy an agent](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-agent.html)
- Teste e troubleshooting de agentes Bedrock: [Test and troubleshoot agent behavior](https://docs.aws.amazon.com/bedrock/latest/userguide/agents-test.html)
- Permissões IAM para agentes e colaboração: [Permissions for Amazon Bedrock Agents](https://docs.aws.amazon.com/bedrock/latest/userguide/agents-permissions.html)
